In [1]:
import torch
import numpy as np
from scipy.stats import t
import torch.nn.functional as F

import os
os.chdir("/workspace")

os.environ["CUDA_VISIBLE_DEVICES"] = "1"

# cumsum 속도 측정

In [68]:
def cumsum_speed(seq_length, device):
    a = torch.full((seq_length, ), 1, device=device)
    
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    
    start.record()
    a = a.cumsum(dim=0)
    end.record()
    
    torch.cuda.synchronize()
    rec_time = start.elapsed_time(end)
    return rec_time

## seq 128

In [69]:
sample_num = 10000
samples = []
cur_seq = 128

for i in range(sample_num):
    st = cumsum_speed(cur_seq, "cuda:1")
    
    samples.append(st)

In [70]:
dof = sample_num - 1
m = np.mean(samples)
sample_se = np.std(samples, ddof=1)
se = sample_se / 10

print(t.interval(0.95, dof, loc=m, scale=se))

(0.003260587456976876, 0.01573621867586312)


## seq 256

In [71]:
sample_num = 10000
samples = []
cur_seq = 256

for i in range(sample_num):
    st = cumsum_speed(cur_seq, "cuda:1")
    
    samples.append(st)

In [72]:
dof = sample_num - 1
m = np.mean(samples)
sample_se = np.std(samples, ddof=1)
se = sample_se / 10

print(t.interval(0.95, dof, loc=m, scale=se))

(0.003482617941666874, 0.01606185547568089)


## seq 512

In [73]:
sample_num = 10000
samples = []
cur_seq = 512

for i in range(sample_num):
    st = cumsum_speed(cur_seq, "cuda:1")
    
    samples.append(st)

In [74]:
dof = sample_num - 1
m = np.mean(samples)
sample_se = np.std(samples, ddof=1)
se = sample_se / 10

print(t.interval(0.95, dof, loc=m, scale=se))

(0.004250784603546508, 0.014832242395060001)


## seq 1024

In [75]:
sample_num = 10000
samples = []
cur_seq = 1024

for i in range(sample_num):
    st = cumsum_speed(cur_seq, "cuda:1")
    
    samples.append(st)

In [76]:
dof = sample_num - 1
m = np.mean(samples)
sample_se = np.std(samples, ddof=1)
se = sample_se / 10

print(t.interval(0.95, dof, loc=m, scale=se))

(0.0013530131529841255, 0.01828237861228998)


# minGRU 한 개의 seq 길이별 성능

In [2]:
from minGRU import minGRU

model = minGRU(128, 2)
model.to("cuda")
torch.compile(model, mode="reduce-overhead")

OptimizedModule(
  (_orig_mod): minGRU(
    (to_hidden_and_gate): Linear(in_features=128, out_features=512, bias=False)
    (to_out): Linear(in_features=256, out_features=128, bias=False)
  )
)

In [3]:
def mingru_speed(model, batch, seq_len, dim, device):
    x = torch.zeros((batch, seq_len, dim), device=device)
    
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    
    start.record()
    y = model(x)
    
    end.record()
    
    model.zero_grad()
    
    torch.cuda.synchronize()
    
    del x
    del y
    rec_time = start.elapsed_time(end)
    return rec_time

In [4]:
model_dim = 128
batch = 256

In [6]:
sample_num = 4000
samples = []
cur_seq = 128

for i in range(sample_num):
    st = mingru_speed(model, batch, cur_seq, model_dim, "cuda")
    
    samples.append(st)

In [8]:
dof = sample_num - 1
m = np.mean(samples)
sample_se = np.std(samples, ddof=1)
se = sample_se / 10

print(t.interval(0.95, dof, loc=m, scale=se))

(0.2064035733574982, 0.30019170616415985)


In [10]:
sample_num = 4000
samples = []
cur_seq = 256

for i in range(sample_num):
    st = mingru_speed(model, batch, cur_seq, model_dim, "cuda")
    
    samples.append(st)

In [11]:
dof = sample_num - 1
m = np.mean(samples)
sample_se = np.std(samples, ddof=1)
se = sample_se / 10

print(t.interval(0.95, dof, loc=m, scale=se))

(2.9612009028639963, 2.9661100569519827)


In [12]:
sample_num = 4000
samples = []
cur_seq = 512

for i in range(sample_num):
    st = mingru_speed(model, batch, cur_seq, model_dim, "cuda")
    
    samples.append(st)

In [13]:
dof = sample_num - 1
m = np.mean(samples)
sample_se = np.std(samples, ddof=1)
se = sample_se / 10

print(t.interval(0.95, dof, loc=m, scale=se))

(5.87517723581813, 5.881734989428299)


In [14]:
sample_num = 4000
samples = []
cur_seq = 1024

for i in range(sample_num):
    st = mingru_speed(model, batch, cur_seq, model_dim, "cuda")
    
    samples.append(st)

In [15]:
dof = sample_num - 1
m = np.mean(samples)
sample_se = np.std(samples, ddof=1)
se = sample_se / 10

print(t.interval(0.95, dof, loc=m, scale=se))

(11.751618760413015, 11.760194266492045)


In [14]:
sample_num = 4000
samples = []
cur_seq = 2048

for i in range(sample_num):
    st = mingru_speed(model, batch, cur_seq, model_dim, "cuda")
    
    samples.append(st)

In [15]:
dof = sample_num - 1
m = np.mean(samples)
sample_se = np.std(samples, ddof=1)
se = sample_se / 10

print(t.interval(0.95, dof, loc=m, scale=se))

(9.446968600457314, 10.06489191926092)


hidden, gate = self.to_hidden_and_gate(x).chunk(2, dim = -1) 연산은 to_hidden_and_gate를 함수 내에서 직접 선언하여 사용하면 선형이 아님

그런데 모델 내에 선언된 애를 불러서 하도록 하였더니 gpu 시간이 2배 늘어나고 512에서 1024로 넘어갈 때 연산 시간이 10배로 증가하는 문제 발견.

아마 모델이 특정 구간을 넘어가면 메모리 오버헤드 등으로 인해 이런 문제가 발생하는 것으로 보임

# GRU의 시퀀스 길이 별 성능

In [11]:
from customGRU import GRUModel

gru = GRUModel(64, 128, 1, 128)
gru.to("cuda")

GRUModel(
  (gru_cell): GRUCell(
    (x2h): Linear(in_features=64, out_features=384, bias=True)
    (h2h): Linear(in_features=128, out_features=384, bias=True)
  )
  (fc): Linear(in_features=128, out_features=128, bias=True)
)

In [12]:
def gru_speed(model, batch, seq_len, dim, device):
    x = torch.zeros((batch, seq_len, dim), device=device)
    
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    
    start.record()
    y = model(x)
    end.record()
    
    model.zero_grad()
    
    torch.cuda.synchronize()
    
    del x
    rec_time = start.elapsed_time(end)
    return rec_time

In [13]:
model_dim = 64
batch = 256

In [15]:
sample_num = 4000
samples = []
cur_seq = 128

for i in range(sample_num):
    st = gru_speed(gru, batch, cur_seq, model_dim, "cuda")
    
    samples.append(st)

In [16]:
dof = sample_num - 1
m = np.mean(samples)
sample_se = np.std(samples, ddof=1)
se = sample_se / 10

print(t.interval(0.95, dof, loc=m, scale=se))

(17.593885157151135, 18.260239801840868)


In [17]:
sample_num = 1000
samples = []
cur_seq = 512

for i in range(sample_num):
    st = gru_speed(gru, batch, cur_seq, model_dim, "cuda")
    
    samples.append(st)

In [18]:
dof = sample_num - 1
m = np.mean(samples)
sample_se = np.std(samples, ddof=1)
se = sample_se / 10

print(t.interval(0.95, dof, loc=m, scale=se))

(71.74862283072534, 74.6957423273557)


In [13]:
sample_num = 4000
samples = []
cur_seq = 1024

for i in range(sample_num):
    st = gru_speed(gru, batch, cur_seq, model_dim, "cuda")
    
    samples.append(st)

In [14]:
dof = sample_num - 1
m = np.mean(samples)
sample_se = np.std(samples, ddof=1)
se = sample_se / 10

print(t.interval(0.95, dof, loc=m, scale=se))

(3.509265715036898, 4.071084195977593)


In [15]:
sample_num = 4000
samples = []
cur_seq = 2048

for i in range(sample_num):
    st = gru_speed(gru, batch, cur_seq, model_dim, "cuda")
    
    samples.append(st)

In [16]:
dof = sample_num - 1
m = np.mean(samples)
sample_se = np.std(samples, ddof=1)
se = sample_se / 10

print(t.interval(0.95, dof, loc=m, scale=se))

(7.8228291635066265, 8.895390422342489)


# Heinsen 알고리즘 자체 속도

In [2]:
def compute_in_parallel_special_case(coeffs, values):
    log_coeffs = torch.log(coeffs)
    log_values = torch.log(values)
    a_star = F.pad(torch.cumsum(log_coeffs, dim=-1), (1, 0))              # eq (2) in paper
    log_x0_plus_b_star = torch.logcumsumexp(log_values - a_star, dim=-1)  # eq (7) in paper
    log_x = a_star + log_x0_plus_b_star                                   # eq (1) in paper
    return torch.exp(log_x)                                               # already a float

def heinsen_speed(coeffs, values):
    
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    
    start.record()
    x = compute_in_parallel_special_case(coeffs, values)
    end.record()
    
    torch.cuda.synchronize()
    
    del x
    rec_time = start.elapsed_time(end)
    return rec_time

## seq 128

In [43]:
batch_size = 256

In [44]:
seq_len = 128  # change as you wish
device = 'cuda'

# Generate some random input data:
coeffs = torch.rand((batch_size, seq_len), device=device) + 1e-6    # eps for numerical stability
values = torch.rand((batch_size, 1 + seq_len), device=device) * 3   # all values >= 0

In [45]:
sample_num = 4000
samples = []

for i in range(sample_num):
    st = heinsen_speed(coeffs, values)
    
    samples.append(st)

In [46]:
dof = sample_num - 1
m = np.mean(samples)
sample_se = np.std(samples, ddof=1)
se = sample_se / 10

print(t.interval(0.95, dof, loc=m, scale=se))

(0.008222304680649217, 0.026628286675343726)


## seq 512

In [47]:
seq_len = 512  # change as you wish
device = 'cuda'

# Generate some random input data:
coeffs = torch.rand((batch_size, seq_len), device=device) + 1e-6    # eps for numerical stability
values = torch.rand((batch_size, 1 + seq_len), device=device) * 3   # all values >= 0

In [48]:
sample_num = 4000
samples = []

for i in range(sample_num):
    st = heinsen_speed(coeffs, values)
    
    samples.append(st)

In [49]:
dof = sample_num - 1
m = np.mean(samples)
sample_se = np.std(samples, ddof=1)
se = sample_se / 10

print(t.interval(0.95, dof, loc=m, scale=se))

(0.008818413155156708, 0.025121170277296514)


## seq 1024

In [50]:
seq_len = 1024  # change as you wish
device = 'cuda'

# Generate some random input data:
coeffs = torch.rand((batch_size, seq_len), device=device) + 1e-6    # eps for numerical stability
values = torch.rand((batch_size, 1 + seq_len), device=device) * 3   # all values >= 0

In [51]:
sample_num = 4000
samples = []

for i in range(sample_num):
    st = heinsen_speed(coeffs, values)
    
    samples.append(st)

In [52]:
dof = sample_num - 1
m = np.mean(samples)
sample_se = np.std(samples, ddof=1)
se = sample_se / 10

print(t.interval(0.95, dof, loc=m, scale=se))

(0.006974399567741976, 0.023947711940257965)


## seq 2048

In [53]:
seq_len = 2048  # change as you wish
device = 'cuda'

# Generate some random input data:
coeffs = torch.rand((batch_size, seq_len), device=device) + 1e-6    # eps for numerical stability
values = torch.rand((batch_size, 1 + seq_len), device=device) * 3   # all values >= 0

In [54]:
sample_num = 4000
samples = []

for i in range(sample_num):
    st = heinsen_speed(coeffs, values)
    
    samples.append(st)

In [55]:
dof = sample_num - 1
m = np.mean(samples)
sample_se = np.std(samples, ddof=1)
se = sample_se / 10

print(t.interval(0.95, dof, loc=m, scale=se))

(0.008550572480397704, 0.025231442893183385)


확실히 heisen sum은 선형이 아니네